# Machine Learning for Applied Research

## 4: Model Training in Practice

In this notebook, we will cover:
- How linear models work, and what their coefficients mean
- How decision trees split data, and why they tend to overfit unless constrained
- How ensemble methods like Random Forest combine many trees to reduce variance
- A structured, fair process for training and comparing models

### Introduction

Notebook 3 established why we split data and how model complexity trades off bias against variance. Now we turn to the model families you will likely use: linear models, decision trees, and ensembles. Understanding roughly how each one works will help you choose sensibly and interpret results critically in your own research, rather than treating every model as an interchangeable black box.

### Setup: Reload the Data

In [ ]:
!pip install ucimlrepo --quiet

from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

energy = fetch_ucirepo(id=242)
X = energy.data.features
y = energy.data.targets

X.columns = ["Relative Compactness",
             "Surface Area",
             "Wall Area",
             "Roof Area",
             "Overall Height",
             "Orientation",
             "Glazing Area",
             "Glazing Area Distribution"]

y.columns = ["Heating Load", "Cooling Load"]

target_col = y.columns[0]   # Heating Load
df = pd.concat([X, y[[target_col]]], axis=1)

X_train, X_test, y_train, y_test = train_test_split(
    df.drop(columns=[target_col]), df[target_col], test_size=0.25, random_state=42
)

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

print(f"Train: {X_train.shape[0]} rows | Test: {X_test.shape[0]} rows")

### Linear Models

A linear model predicts the target as a weighted sum of the features, plus a constant offset:

ŷ = w₀ + w₁·x₁ + w₂·x₂ + ... + wₚ·xₚ

Fitting the model means choosing the weights (w₁...wₚ) and intercept (w₀) that minimize the total squared error between predictions and actual values on the training data. This method is called *ordinary least squares*. Unlike most other ML methods, this has a closed-form solution: no iterative search is needed, which is part of why linear models are fast and deterministic to train.

The biggest practical advantage of a linear model is **interpretability**: each coefficient tells you how much the prediction changes for a one-unit increase in that feature, holding all others constant. That is often exactly what applied researchers need to report, rather than just a black-box prediction.

The biggest limitation: a *linear* model can only capture *linear* relationships, unless you engineer additional features by hand (as we did with polynomial features in Notebook 3).

#### 1.1 Exercises:
> (a) Import `LinearRegression` from `sklearn.linear_model` and initialize it as `linreg`.
>
> (b) Train (`fit`) the model on `X_train` and `y_train`.
>
> (c) Use the trained model to `predict` values for `X_train`, and calculate and print the RMSE for `y_train`.
>
> (d) Do the same for `X_test` and `y_test`.

Beyond overall error, we can look at exactly what the model learned by inspecting its coefficients.

> (e) Build a `pandas.Series` of `linreg.coef_`, indexed by `X_train.columns`, and sort it by absolute value in descending order.
>
> (f) Print the intercept (`linreg.intercept_`) as well.
>
> (g) Which feature has the largest-magnitude coefficient? In one or two sentences, interpret what its sign means for heating load and note whether you would trust the magnitude alone to call it the "most important" feature (hint: think about the units and scale of each feature).

In [ ]:
# Your solution:





##### Solution:


In [ ]:
from sklearn.linear_model import LinearRegression

linreg = LinearRegression()
linreg.fit(X_train, y_train)

print(f"Linear Regression - Train RMSE: {rmse(y_train, linreg.predict(X_train)):.2f}")
print(f"Linear Regression - Test RMSE: {rmse(y_test, linreg.predict(X_test)):.2f}")

coefficients = pd.Series(linreg.coef_, index=X_train.columns).sort_values(key=abs, ascending=False)
print("Coefficients (sorted by magnitude):")
print(coefficients)
print(f"\nIntercept: {linreg.intercept_:.2f}")

# (g) The largest-magnitude coefficient is typically on one of the
# small-scale features (e.g. Relative Compactness, which only ranges
# roughly 0.6-1.0) rather than one measured in square meters. A large
# coefficient on a small-scale feature and a small coefficient on a
# large-scale feature can have similar real-world impact on the
# prediction. So magnitude alone, without accounting for each feature's
# scale, is misleading. Comparing coefficients after scaling all features
# to the same range would be a fairer way to judge relative importance.

### Decision Trees

A decision tree predicts by recursively splitting the data into groups based on threshold questions about individual features - for example, "is Overall Height less than 5 meters?" At each split, the tree picks the question that most reduces the error within the resulting groups (for regression, this usually means reducing the variance of the target within each group). At prediction time, a new example is routed down the tree by answering these questions in sequence until it reaches a leaf, and the tree's prediction is the average target value of the training examples that ended up in that leaf.

<img src="https://drive.google.com/uc?id=1QwVCzPWK1lv5tJvjESUKb-iqj3KPTybp" style = "height:350px"> <br>

Trees have two attractive properties: they capture nonlinear relationships and interactions between features automatically, without you having to engineer polynomial terms by hand, and they require no feature scaling at all - splits only care about the order of values, not their absolute scale.

Left unconstrained, however, a tree will keep splitting until every leaf contains just one training example - perfectly fitting, and memorizing, the training data, including its noise. This is the same high-variance behavior we saw with high-degree polynomials in Notebook 3. Limiting how deep a tree is allowed to grow (`max_depth`), or how many examples a leaf must contain, trades away some training-set accuracy for better generalization.



In [ ]:
from sklearn.tree import DecisionTreeRegressor

for depth in [3, None]:
    tree = DecisionTreeRegressor(max_depth=depth, random_state=42)
    tree.fit(X_train, y_train)
    label = f"max_depth={depth}" if depth else "no depth limit"
    print(f"{label:20s} - Train RMSE: {rmse(y_train, tree.predict(X_train)):6.2f} | Test RMSE: {rmse(y_test, tree.predict(X_test)):6.2f}")

#### 1.2 Exercises:
> (a) Loop over `max_depth` values from 1 to 15. For each, fit a `DecisionTreeRegressor` (`random_state=42`) and record its train and test RMSE.
>
> (b) Plot both RMSE curves against `max_depth` on a single line chart (you will need to import `matplotlib`).
>
> (c) At roughly which depth does the test RMSE stop improving, or start getting worse? What does that tell you about where bias gives way to variance for this dataset?

In [ ]:
# Your solution:





##### Solution:

In [ ]:
import matplotlib.pyplot as plt

depths = range(1, 16)
train_errors = []
test_errors = []

for depth in depths:
    tree = DecisionTreeRegressor(max_depth=depth, random_state=42)
    tree.fit(X_train, y_train)
    train_errors.append(rmse(y_train, tree.predict(X_train)))
    test_errors.append(rmse(y_test, tree.predict(X_test)))

plt.figure(figsize=(8, 5))
plt.plot(list(depths), train_errors, marker="o", label="Train RMSE")
plt.plot(list(depths), test_errors, marker="o", label="Test RMSE")
plt.xlabel("max_depth")
plt.ylabel("RMSE")
plt.title("Decision tree: train vs. test RMSE by depth")
plt.legend()
plt.show()

best_depth = list(depths)[int(np.argmin(test_errors))]
print(f"Depth with lowest test RMSE: {best_depth}")

# (c) Train RMSE keeps dropping as depth increases.
# The tree can always fit the training data better with more splits 
# but test RMSE flattens out and then creeps back up past a certain depth. 
# That turning point is where the tree starts fitting noise specific 
# to the training set rather than the underlying relationship.

### Ensemble Methods: Random Forest

A **Random Forest** is an *ensemble* of many decision trees. Each tree is trained on a random bootstrap sample of the training data (sampling with replacement), and at each split, the tree is only allowed to consider a random subset of the features. The forest's final prediction is the average of all its trees' individual predictions.

Why does this help? Each individual tree, especially a deep unconstrained one, has high variance, so it fits its particular bootstrap sample very closely. But because each tree sees a different, randomized version of the data, their errors tend to be somewhat independent of each other. Averaging many such trees cancels out much of that variance while keeping the low bias that comes from letting each tree grow deep. This combination (many high-variance, low-bias trees averaged) is why Random Forests are often a strong out-of-the-box choice for tabular data, with comparatively little tuning effort.

An alternative ensemble strategy, **gradient boosting**, builds trees sequentially instead of in parallel: each new tree is trained specifically to correct the errors of the trees built so far, rather than being averaged with independent copies. *Boosted ensembles* are often even more accurate than Random Forests, but are more sensitive to their settings and easier to overfit if not tuned carefully.

#### 1.3 Exercises:
> (a) Import `RandomForestRegressor` from the `ensemble` submodule of `sklearn`.
>
> Random Forests have a parameter called `n_estimators`, which means the number of trees in the forest.
>
> (b) Using a loop, fit a `RandomForestRegressor` with `[1, 5, 10, 50, 200, 500]` trees. Use a maximum depth of 6 and random state of 42. Record the test RMSE.
>
> (c) Plot test RMSE against `n_estimators`, using a log scale on the x-axis (`plt.xscale("log")`).
>
> (d) In one sentence, describe what happens to test RMSE as you keep adding trees past a certain point.

In [ ]:
# Your solution:




##### Solution:

In [ ]:
from sklearn.ensemble import RandomForestRegressor

n_values = [1, 5, 10, 50, 200, 500]
forest_test_errors = []

for n in n_values:
    forest_n = RandomForestRegressor(n_estimators=n, max_depth=6, random_state=42)
    forest_n.fit(X_train, y_train)
    forest_test_errors.append(rmse(y_test, forest_n.predict(X_test)))

plt.figure(figsize=(8, 5))
plt.plot(n_values, forest_test_errors, marker="o")
plt.xscale("log")
plt.xlabel("n_estimators (log scale)")
plt.ylabel("Test RMSE")
plt.title("Random Forest: test RMSE vs. number of trees")
plt.show()

for n, err in zip(n_values, forest_test_errors):
    print(f"n_estimators={n:4d} - Test RMSE: {err:.2f}")

# (d) Test RMSE drops quickly as trees are added at first, then flattens
# out. Past a certain point, more trees mostly cost extra computation
# time rather than buying meaningfully better accuracy.

### Training Processes and Model Evaluation

With three model families in hand, it is worth stating explicitly the process that makes comparing them fair:

1. Split the data once, before touching any model! Every model below is trained and evaluated on the exact same rows.
2. Fit each candidate model on the same training set.
3. Evaluate each on the same held-out test set, using the same metric.
4. Compare using that metric, but also weigh practical factors that a single number will not capture: interpretability, training time, and how sensitive the model is to its settings.

One easy-to-miss trap: any random behavior such as the train/test split itself or a Random Forest's bootstrap sampling should be pinned down with `random_state`, as we have been doing throughout. Without it, rerunning the same code could quietly produce a different split or different trees, making two runs impossible to compare fairly. A related and more serious version of this problem is data leakage (i.e., accidentally letting information from the test set influence training), which we will look at directly in the coming notebooks.

#### 1.4 Exercises:
> (a) Build a summary table (a `DataFrame` works well) comparing the linear model, your best-depth decision tree, and a Random Forest with 200 trees, each with its train and test RMSE.
>
> (b) Based on test RMSE alone, which model would you pick?
>
> (c) In 2-3 sentences, say whether you would pick the same model if interpretability mattered as much as accuracy for your research context, and name which model you would choose in that case.

In [ ]:
# Your solution:




##### Solution:

In [ ]:
best_tree = DecisionTreeRegressor(max_depth=best_depth, random_state=42)
best_tree.fit(X_train, y_train)

forest = RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42)
forest.fit(X_train, y_train)

summary = pd.DataFrame({
    "Model": ["Linear Regression", f"Decision Tree (depth={best_depth})", "Random Forest"],
    "Train RMSE": [
        rmse(y_train, linreg.predict(X_train)),
        rmse(y_train, best_tree.predict(X_train)),
        rmse(y_train, forest.predict(X_train)),
    ],
    "Test RMSE": [
        rmse(y_test, linreg.predict(X_test)),
        rmse(y_test, best_tree.predict(X_test)),
        rmse(y_test, forest.predict(X_test)),
    ],
})
display(summary)

# (b) Based on test RMSE alone, the Random Forest is typically the
# strongest performer of the three here.

# (c) If interpretability mattered as much as accuracy (for example, if
# you needed to explain to a building-code reviewer exactly why a design
# was flagged) the linear model's directly readable coefficients would
# likely be worth an accuracy trade-off. A single decision tree sits in
# between: still visualizable and explainable as a set of if/else rules,
# though less transparent than a linear model's coefficients.

#### Challenge (optional):
> Fit a `GradientBoostingRegressor(random_state=42)` with default settings on the same training data. How does its test RMSE compare to the Random Forest? Unlike Random Forest, gradient boosting builds its trees sequentially rather than averaging independent ones. Do you remember: Was it more or less sensitive to its exact settings?

In [ ]:
# Your solution:




##### Solution:

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

boosted = GradientBoostingRegressor(random_state=42)
boosted.fit(X_train, y_train)

print(f"Gradient Boosting - Train RMSE: {rmse(y_train, boosted.predict(X_train)):.2f} | Test RMSE: {rmse(y_test, boosted.predict(X_test)):.2f}")

# Gradient boosting often matches or slightly beats the Random Forest here
# with zero tuning, though it can also overfit more easily: each tree is
# built specifically to correct the previous ones' mistakes, so there is
# no averaging-out of an individual tree's bad decisions the way there is
# in a Random Forest. This sensitivity is exactly why boosted models
# usually need more careful hyperparameter tuning in practice.

## Conclusion

We have now seen various model families:

- **Linear models**: fast, deterministic, and directly interpretable through their coefficients, but limited to linear relationships
- **Decision trees**: capture nonlinearity and interactions automatically, need no feature scaling, but overfit badly unless their depth is constrained
- **Random Forests**: average many high-variance trees to cancel out much of that variance, usually a strong default choice for tabular data
- **A fair comparison process**: same split, same metric, `random_state` pinned down throughout
- **Gradient boosting**: a sequential alternative to Random Forest's parallel averaging, often more accurate but more tuning-sensitive